In [6]:
import os, csv, json, shutil, hashlib, math
from collections import Counter

# ================= SETTINGS =================
LAST_NAME  = "Pacudan"
STUDENT_ID = "TUPM-26-1859"
CSV_FILE   = "Global_Cybersecurity_Threats_2015-2024.csv"
# ============================================

SEED_DIGIT  = int(STUDENT_ID[-1])
TARGET_YEAR = 2015 + SEED_DIGIT
PREFIX      = f"{LAST_NAME}_{STUDENT_ID}"

LOSS_COL, TYPE_COL, YEAR_COL = "Financial Loss (in Million $)", "Attack Type", "Year"
COUNTRY_COL, INDUSTRY_COL = "Country", "Target Industry"

os.makedirs("output", exist_ok=True)
os.makedirs("backup", exist_ok=True)

FILTERED_CSV = os.path.join("output", f"{PREFIX}_incidents_filtered.csv")
JSON_FILE    = os.path.join("output", f"{PREFIX}_incidents_summary.json")
BACKUP_FILE  = os.path.join("backup", f"{PREFIX}_incidents_summary.json")
REPORT_FILE  = os.path.join("output", f"{PREFIX}_incident_report.txt")

def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def to_year(v):
    try:
        return int(float(str(v).strip()))
    except (ValueError, TypeError):
        return None

def to_loss(v):
    """Valid non-negative float, or None if invalid."""
    try:
        x = float(str(v).strip())
    except (ValueError, TypeError):
        return None
    if math.isnan(x) or math.isinf(x) or x < 0:
        return None
    return x

print("Working folder (put the CSV here):", os.getcwd(), "\n")

if not os.path.exists(CSV_FILE):
    print(f"ERROR: '{CSV_FILE}' not found in the working folder.")
    print("Files here:", os.listdir("."))
    print("\n=== FINAL OUTPUT ===\nFAILED: CSV file missing.")
else:
    # ---- Read data (original order preserved) ----
    with open(CSV_FILE, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        reader.fieldnames = [c.strip() for c in reader.fieldnames]
        fieldnames = reader.fieldnames
        rows = list(reader)

    missing = [c for c in (LOSS_COL, TYPE_COL, YEAR_COL) if c not in fieldnames]
    if missing:
        print("ERROR: missing columns:", missing, "\nFound:", fieldnames)
        print("\n=== FINAL OUTPUT ===\nFAILED: bad columns.")
    else:
        total_records = len(rows)

        # ---- Select target year ----
        selected = [(i, r) for i, r in enumerate(rows) if to_year(r[YEAR_COL]) == TARGET_YEAR]
        incidents = len(selected)

        # ---- Count per attack type ----
        counts = Counter(((r[TYPE_COL] or "").strip() or "Unknown") for _, r in selected)
        type_counts = dict(counts)

        # ---- Valid / invalid losses ----
        valid, invalid_count = [], 0
        for i, r in selected:
            loss = to_loss(r[LOSS_COL])
            if loss is None:
                invalid_count += 1
            else:
                valid.append((i, loss, r))

        total_loss = round(sum(l for _, l, _ in valid), 2) if valid else 0.0
        avg_loss   = round(total_loss / len(valid), 2) if valid else 0.0

        # ---- Five costliest (desc loss, ties keep original order) ----
        top5 = sorted(valid, key=lambda t: (-t[1], t[0]))[:5]
        top5_records = [
            {"country": r.get(COUNTRY_COL), "attack_type": r.get(TYPE_COL),
             "target_industry": r.get(INDUSTRY_COL), "financial_loss": l}
            for _, l, r in top5
        ]

        # ---- Filtered CSV ----
        with open(FILTERED_CSV, "w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            w.writeheader()
            for _, r in selected:
                w.writerow(r)

        # ---- JSON summary ----
        summary = {
            "target_year": TARGET_YEAR,
            "total_records": total_records,
            "incidents_in_target_year": incidents,
            "invalid_records": invalid_count,
            "count_per_attack_type": type_counts,
            "total_financial_loss_usd_millions": total_loss,
            "average_financial_loss_usd_millions": avg_loss,
            "five_costliest_incidents": top5_records,
        }
        with open(JSON_FILE, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=4)

        # ---- Backup + verification ----
        shutil.copy2(JSON_FILE, BACKUP_FILE)
        backup_exists = os.path.exists(BACKUP_FILE)
        backup_ok = backup_exists and sha256(JSON_FILE) == sha256(BACKUP_FILE)

        # ---- Final output sentence ----
        if incidents == 0:
            final = f"No incidents were found for {TARGET_YEAR}."
        elif not backup_ok:
            final = "FAILED: backup verification did not match."
        else:
            final = (f"{incidents} incidents occurred in {TARGET_YEAR}, "
                     f"with a total loss of ${total_loss} million.")

        # ---- Build report (same text is printed and saved) ----
        L = [
            f"Target year: {TARGET_YEAR}",
            f"Total records in dataset: {total_records}",
            f"Incidents in target year: {incidents}",
            f"Invalid records: {invalid_count}",
            f"Count per attack type: {type_counts}",
            f"Total financial loss (USD millions): {total_loss}",
            f"Average financial loss (USD millions): {avg_loss}",
            "Five costliest incidents:",
        ]
        L += [f"  {rec}" for rec in top5_records] if top5_records else ["  (none)"]
        L += [
            f"Filtered CSV file: {FILTERED_CSV}",
            f"JSON summary file: {JSON_FILE}",
            f"Backup verification: {BACKUP_FILE} | exists: {backup_exists} | hash match: {backup_ok}",
            f"Report file: {REPORT_FILE}",
            "",
            "=== FINAL OUTPUT ===",
            final,
            f"Report saved to {os.path.basename(REPORT_FILE)}",
        ]
        report_text = "\n".join(L)
        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            f.write(report_text)

        print(report_text)

Working folder (put the CSV here): c:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises 

Target year: 2024
Total records in dataset: 3000
Incidents in target year: 299
Invalid records: 0
Count per attack type: {'Ransomware': 53, 'DDoS': 54, 'Phishing': 62, 'Malware': 43, 'Man-in-the-Middle': 47, 'SQL Injection': 40}
Total financial loss (USD millions): 15434.29
Average financial loss (USD millions): 51.62
Five costliest incidents:
  {'country': 'China', 'attack_type': 'DDoS', 'target_industry': 'Banking', 'financial_loss': 99.99}
  {'country': 'Russia', 'attack_type': 'SQL Injection', 'target_industry': 'Education', 'financial_loss': 99.31}
  {'country': 'Japan', 'attack_type': 'DDoS', 'target_industry': 'Healthcare', 'financial_loss': 99.28}
  {'country': 'France', 'attack_type': 'Ransomware', 'target_industry': 'Retail', 'financial_loss': 99.03}
  {'country': 'France', 'attack_type': 'Phishing', 'target_industry': 'Retail', 'financial_loss': 98.95}
Filtered CSV file: 